# Диагностика, ранняя остановка и сохранение модели

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Читать кривые обучения, выбирать раннюю остановку и сохранять независимое состояние модели.

## Записываем наблюдения, а не впечатления

После каждой эпохи записывают потерю на train, метрику на validation, размер шага и число просмотренных объектов. Если обучающая потеря уменьшается, а качество на validation ухудшается, дальнейшее обучение может усиливать переобучение. Сначала, однако, проверьте режимы модели, согласованность функции потерь и разбиение данных.

Ранняя остановка выбирает состояние по validation. Patience — число разрешённых проверок без улучшения; правило, направление метрики и минимальное изменение задают заранее. Итоговый тест не используется как сигнал остановки.

## Разобранный пример: лучший не последний

Потеря на validation по эпохам [0.8,0.5,0.4,0.45,0.47]. Минимум на третьей эпохе. Если остановились на пятой, для выбранного решения нужно восстановить третье состояние, а не просто сохранить последние веса. Присваивание best=model не создаёт копии, поэтому последующие обновления изменят и «лучший» объект.

`state_dict` — словарь параметров и другого состояния модели, например накопленных статистик слоёв. `deepcopy` создаёт независимую копию словаря, `load_state_dict` восстанавливает состояние модели из него. Для записи во временный файл и чтения используем `torch.save` и `torch.load` с `weights_only=True`; загружаем только файл, который создали сами.

## Диагностика бюджета

Проверяйте время эпохи, память и качество на небольшой части данных до большого запуска. Если простая сеть не может уменьшить ошибку на нескольких объектах, более крупная сеть редко исправляет ошибки форм массивов или меток. Для продолжения обучения сохраняют также оптимизатор, номер эпохи и настройки; для предсказания достаточно архитектуры и подходящего состояния модели.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

Сначала выберем лучшую эпоху по приведённому списку потерь на validation. `enumerate(..., 1)` нумерует эпохи с единицы; `without_improvement` считает подряд проверки без улучшения. При равенстве оставляем более раннюю эпоху.

In [ ]:
validation_losses = [0.8, 0.5, 0.4, 0.45, 0.47]
best_loss = None
best_epoch = None
without_improvement = 0
patience = 2
for epoch, value in enumerate(validation_losses, 1):
    if best_loss is None or value < best_loss:
        best_loss = value
        best_epoch = epoch
        without_improvement = 0
    else:
        without_improvement += 1
    if without_improvement >= patience:
        break
assert best_epoch == 3
assert epoch == 5
print("Лучшая эпоха:", best_epoch, "Остановка:", epoch)


Далее отдельно проверим копирование и восстановление состояния: сохраним текущие веса, намеренно изменим их и вернём сохранённые. Список оценок выше задан вручную; он не измерен для этой маленькой модели. В настоящем цикле независимую копию состояния делают внутри ветки улучшения validation.

In [ ]:
import copy
from pathlib import Path
from tempfile import TemporaryDirectory
import torch
from torch import nn
torch.set_num_threads(1)
torch.manual_seed(17)
model = nn.Linear(1, 1)
x = torch.tensor([[2.]])
best = copy.deepcopy(model.state_dict())
with torch.no_grad():
    expected = model(x).clone()
    model.weight.add_(1.)
with TemporaryDirectory() as folder:
    path = Path(folder) / "own-weights.pt"
    torch.save(best, path)
    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
model.eval()
with torch.no_grad():
    assert torch.allclose(model(x), expected)
print("Собственное состояние восстановлено")


## Типичные ошибки

Не храните ссылку вместо копии лучшего состояния. Не загружайте неизвестные чужие файлы ради этого примера.

## Задачи

### 1. Базовая

По какой части выбирается ранняя остановка?

<details>
<summary>Подсказка</summary>

Это данные разработки.

</details>



### 2. Базовая

Создаёт ли best=model независимую модель?

<details>
<summary>Подсказка</summary>

Это новое имя того же объекта.

</details>



### 3. Прикладная

В [0.8,0.5,0.4,0.45] выберите лучшую эпоху.

<details>
<summary>Подсказка</summary>

Минимизируем loss.

</details>



### 4. Прикладная

Patience=2: после минимума два ухудшения. Что сделать?

<details>
<summary>Подсказка</summary>

Следуйте правилу и восстановите лучший момент.

</details>



### 5. Олимпиадная

Почему одинаковое имя переменной не гарантирует сохранение лучших весов?

<details>
<summary>Подсказка</summary>

Объект мог продолжать меняться.

</details>



### 6. Олимпиадная

Почему почти нулевая ошибка на пяти обучающих объектах не заменяет итоговую проверку?

<details>
<summary>Подсказка</summary>

Это диагностический тест способности учиться.

</details>



## Итоги и самопроверка

Объясните, какие данные хранит сохранённое состояние модели и почему восстановление лучшей эпохи отличается от остановки на последней.

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

По validation, не по финальному тесту.

### Задача 2

Нет. Нужна копия состояния.

### Задача 3

Третья, значение 0.4.

### Задача 4

Остановить обучение и использовать сохранённое лучшее состояние, а не обязательно последнее.

### Задача 5

Имена и ссылки не фиксируют значения во времени. Нужна независимая копия данных.

### Задача 6

Он проверяет механику, а качество требует новых независимых объектов.